# 🌍 Earth Field Analysis
## Layer 3 – Atmosphere / Weather / Thunderstorms

| Layer | Name | Status |
|-------|------|--------|
| 0 | External Cosmic Drivers | ✅ `layer0_state.json` |
| 1 | Planetary Body | ✅ `layer1_state.json` |
| 2 | Surface / Oceans / Land | ✅ `layer2_state.json` |
| **3** | **Atmosphere / Weather / Thunderstorms** | **← this layer** |
| 4 | Ionosphere | ⬜ |
| 5 | Global Electric Circuit | ⬜ |
| 6 | Resonance Field / Schumann | ⬜ |
| 7 | Earth Field State Engine | ⬜ |

> **System logic:** Temperature + Humidity + Dynamics → Convection → Thunderstorms → Lightning → electromagnetic excitation
>
> **Significance:** Layer 3 is the primary driver of Schumann resonance.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import datetime, json, math, re, requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

print(f'Packages loaded')
print(f'Analysis date: {datetime.date.today()}')

# Load previous layers
context = {}
for n in [0, 1, 2]:
    try:
        with open(f'layer{n}_test_state.json', encoding='utf-8') as f:
            context[n] = json.load(f)
        sc = context[n].get('score', '–')
        lv = context[n].get('level', '–').upper()
        print(f'  Layer {n}: {lv:8}  Score={sc}')
    except FileNotFoundError:
        print(f'  Layer {n}: not found')
        context[n] = None

layer0 = context.get(0)
layer1 = context.get(1)
layer2 = context.get(2)

# Thunderstorm trigger from Layer 2
l2_thunder = layer2.get('flags', {}).get('thunderstorm_trigger', False) if layer2 else False
l2_conv    = layer2.get('raw_values', {}).get('conv_potential_mean') if layer2 else None
print(f'  L2 thunderstorm_trigger: {l2_thunder}  conv_potential: {l2_conv}')

Packages loaded
Analysis date: 2026-05-12
  Layer 0: CALM      Score=0.1646
  Layer 1: CALM      Score=0.2924
  Layer 2: MODERATE  Score=0.347
  L2 thunderstorm_trigger: False  conv_potential: 0.477


---
## 1. System Structure: The Elements of Layer 3

In [2]:
elements = [
    {'name': 'Air Pressure &<br>Pressure Gradient', 'x': 0.50, 'y': 0.92, 'color': '#5F5E5A', 'sz': 54},
    {'name': 'Temperature &<br>CAPE',          'x': 0.20, 'y': 0.77, 'color': '#E85D24', 'sz': 54},
    {'name': 'Humidity &<br>Dew Point',         'x': 0.80, 'y': 0.77, 'color': '#378ADD', 'sz': 54},
    {'name': 'Wind &<br>Jet Stream',            'x': 0.10, 'y': 0.54, 'color': '#888780', 'sz': 52},
    {'name': 'Convection &<br>Thunderstorm Cells', 'x': 0.50, 'y': 0.57, 'color': '#7F77DD', 'sz': 58},
    {'name': 'Clouds &<br>OLR',               'x': 0.90, 'y': 0.54, 'color': '#B4B2A9', 'sz': 52},
    {'name': 'Lightning Activity<br>(GEC Input)', 'x': 0.50, 'y': 0.33, 'color': '#F2A623', 'sz': 60},
]
core = {'x': 0.50, 'y': 0.12}

fig = go.Figure()
for el in elements:
    dx = core['x'] - el['x']; dy = core['y'] - el['y']
    dist = math.sqrt(dx**2 + dy**2)
    t = 0.05 / dist
    fig.add_trace(go.Scatter(
        x=[el['x'], core['x'] - dx*t], y=[el['y'], core['y'] - dy*t],
        mode='lines', line=dict(color=el['color'], width=1.6), opacity=0.4,
        showlegend=False, hoverinfo='skip'
    ))

fig.add_trace(go.Scatter(
    x=[core['x']], y=[core['y']], mode='markers+text',
    marker=dict(size=82, color='#534AB7', opacity=0.90, line=dict(color='#C0BBFF', width=2)),
    text=['⛈️ Schumann<br>Exciter'], textposition='middle center',
    textfont=dict(size=10, color='white'), showlegend=False, hoverinfo='skip'
))
for el in elements:
    fig.add_trace(go.Scatter(
        x=[el['x']], y=[el['y']], mode='markers+text',
        marker=dict(size=el['sz'], color=el['color'], opacity=0.88,
                    line=dict(color='white', width=1.8)),
        text=[el['name']], textposition='middle center',
        textfont=dict(size=9.5, color='white'), showlegend=False,
        hovertemplate=el['name'].replace('<br>',' ') + '<extra></extra>'
    ))

for txt, px, py, col in [
    ('🌡️  Thermodynamics',             0.50, 1.00, '#B03010'),
    ('⚡  Convection & Electricity',   0.50, 0.65, '#3A32A0'),
    ('🌩️  GEC Input (Lightning Activity)', 0.50, 0.23, '#B07800'),
]:
    fig.add_annotation(x=px, y=py, text=txt, showarrow=False,
                       xref='paper', yref='paper', font=dict(size=11, color=col))

# Layer-Kontext
ctx_parts = []
if layer2: ctx_parts.append(f'L2: {layer2["level"].upper()}  Thunder-Trigger: {l2_thunder}')
if layer1: ctx_parts.append(f'L1: {layer1["level"].upper()}')
if layer0: ctx_parts.append(f'L0: {layer0["level"].upper()}  Driver: {layer0.get("dominant_driver","–")}')
if ctx_parts:
    fig.add_annotation(x=0.5, y=0.02, text='  |  '.join(ctx_parts), showarrow=False,
                       xref='paper', yref='paper', font=dict(size=10, color='#888780'))

fig.update_layout(
    title=dict(text='Layer 3 – Atmosphere / Weather / Thunderstorms: System Structure', font=dict(size=16)),
    xaxis=dict(showgrid=False, zeroline=False, visible=False, range=[-0.05, 1.05]),
    yaxis=dict(showgrid=False, zeroline=False, visible=False, range=[0.02, 1.05]),
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    height=540, margin=dict(l=20, r=20, t=55, b=20)
)
fig.show()

---
## 2. Fetch Real-Time Data

In [3]:
# ============================================================
# REAL-TIME DATA – FOUR SOURCES
# 1) Open-Meteo     – Atmospheric profiles (pressure, CAPE, CIN, clouds)
# 2) NOAA GFS       – Jet stream / 500hPa geopotential
# 3) WWLLN Proxy    – Lightning activity (via Blitzortung/NOAA GLDN)
# 4) NOAA GOES      – Outgoing Longwave Radiation / thunderstorm cores
# ============================================================

import time

def fetch_with_retry(url, retries=3, wait=20, timeout=15):
    """Fetch URL with automatic retry on failure"""
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f'    ⚠️  Attempt {attempt+1}/{retries} failed: {str(e)[:60]}')
            if attempt < retries - 1:
                print(f'       Retrying in {wait}s...')
                time.sleep(wait)
    print('    ❌  All attempts failed – returning None')
    return None

raw = {}

# --- 1) Open-Meteo: Atmospheric parameters ---
# Same 6 points as Layer 2, extended with CAPE/CIN and clouds
OM_POINTS = [
    {'name': 'Central Europe',  'lat': 48.0,  'lon':  11.0},
    {'name': 'Tropics (Congo)', 'lat': -4.0,  'lon':  23.0},
    {'name': 'Arctic',          'lat': 78.0,  'lon':  15.0},
    {'name': 'Pacific (ITCZ)',  'lat':  5.0,  'lon': -150.0},
    {'name': 'Amazonia',        'lat': -5.0,  'lon': -60.0},
    {'name': 'Sahel',           'lat': 13.0,  'lon':  10.0},
]
raw['atmos'] = []
for pt in OM_POINTS:
    url = (
        f'https://api.open-meteo.com/v1/forecast'
        f'?latitude={pt["lat"]}&longitude={pt["lon"]}'
        f'&current=temperature_2m,relative_humidity_2m,'
        f'surface_pressure,wind_speed_10m,wind_direction_10m,'
        f'wind_gusts_10m,cloud_cover,precipitation,'
        f'rain,showers,weather_code'
        f'&hourly=cape,convective_inhibition,lifted_index,'
        f'cloud_cover,precipitation_probability'
        f'&forecast_hours=6&timezone=UTC'
    )
    d = fetch_with_retry(url, retries=3, wait=20, timeout=15)
    if d is not None:
        d['_point'] = pt
        raw['atmos'].append(d)
        cur  = d.get('current', {})
        hrly = d.get('hourly', {})
        cape_val = hrly.get('cape', [None])[0]
        print(f'  OK {pt["name"]:<18} P={cur.get("surface_pressure","–")} hPa  '
              f'CAPE={cape_val} J/kg  Cloud={cur.get("cloud_cover","–")}%')
    else:
        print(f'  ❌  {pt["name"]:<18} skipped – no data available')

# --- 2) NOAA SWPC: X-Ray ---
d = fetch_with_retry('https://services.swpc.noaa.gov/json/goes/primary/xrays-1-day.json',
                     retries=3, wait=15, timeout=12)
if d is not None:
    raw['xray'] = d
    print(f'  OK NOAA X-Ray          {len(raw["xray"]):>5} entries')
else:
    raw['xray'] = None
    print('  ❌  NOAA X-Ray skipped')

# --- 3) NASA EONET Storms ---
d = fetch_with_retry('https://eonet.gsfc.nasa.gov/api/v3/events?category=severeStorms&status=open&limit=20',
                     retries=3, wait=15, timeout=12)
if d is not None:
    raw['storms'] = d
    n = len(raw['storms'].get('events', []))
    print(f'  OK NASA EONET Storms   {n:>5} active events')
else:
    raw['storms'] = None
    print('  ❌  NASA EONET skipped')

# --- 4) NOAA Kp ---
d = fetch_with_retry('https://services.swpc.noaa.gov/json/planetary_k_index_1m.json',
                     retries=3, wait=15, timeout=10)
if d is not None:
    raw['kp'] = d
    print(f'  OK NOAA Kp             {len(raw["kp"]):>5} entries')
else:
    raw['kp'] = None
    print('  ❌  NOAA Kp skipped')

print(f'\nData fetch: {len(raw["atmos"])}/{len(OM_POINTS)} points loaded')

  OK Central Europe     P=937.0 hPa  CAPE=10.0 J/kg  Cloud=100%


  OK Tropics (Congo)    P=944.9 hPa  CAPE=1610.0 J/kg  Cloud=25%


  OK Arctic             P=944.2 hPa  CAPE=0.0 J/kg  Cloud=95%


  OK Pacific (ITCZ)     P=1006.5 hPa  CAPE=2230.0 J/kg  Cloud=47%


  OK Amazonia           P=1011.6 hPa  CAPE=210.0 J/kg  Cloud=100%


  OK Sahel              P=971.2 hPa  CAPE=0.0 J/kg  Cloud=0%
  OK NOAA X-Ray           2874 entries


  OK NASA EONET Storms       1 active events
  OK NOAA Kp               358 entries

Data fetch: 6/6 points loaded


In [4]:
# ============================================================
# PROCESS DATA
# ============================================================

# --- Atmospheric parameters per measurement point ---
atmos_rows = []
for d in raw['atmos']:
    pt   = d['_point']
    cur  = d.get('current', {})
    hrly = d.get('hourly', {})
    # Most recent CAPE/CIN value (first hour)
    cape = next((v for v in hrly.get('cape', []) if v is not None), None)
    cin  = next((v for v in hrly.get('convective_inhibition', []) if v is not None), None)
    li   = next((v for v in hrly.get('lifted_index', []) if v is not None), None)
    wcode= cur.get('weather_code')
    atmos_rows.append({
        'point':    pt['name'],
        'lat':      pt['lat'],
        'lon':      pt['lon'],
        'pressure': cur.get('surface_pressure'),
        'temp_2m':  cur.get('temperature_2m'),
        'rh':       cur.get('relative_humidity_2m'),
        'wind_10m': cur.get('wind_speed_10m'),
        'wind_dir': cur.get('wind_direction_10m'),
        'cloud':    cur.get('cloud_cover'),
        'precip':   cur.get('precipitation'),
        'rain':     cur.get('rain'),
        'showers':  cur.get('showers'),
        'gusts':    cur.get('wind_gusts_10m'),
        'wcode':    wcode,
        'cape':     cape,
        'cin':      cin,
        'li':       li,
        # WMO weather_code >= 95 = Thunderstorm
        'thunderstorm': (wcode >= 95) if wcode is not None else False,
    })

df_atm = pd.DataFrame(atmos_rows)
for col in ['pressure','temp_2m','rh','wind_10m','cloud','precip','cape','cin','li','z500','w500']:
    if col in df_atm.columns:
        df_atm[col] = pd.to_numeric(df_atm[col], errors='coerce')

print('Atmospheric parameters:')
if df_atm.empty:
    print('  ⚠️  No atmospheric data available – all API calls failed')
else:
    expected_cols = ['point','pressure','temp_2m','cape','cin','li','cloud','thunderstorm']
    available_cols = [c for c in expected_cols if c in df_atm.columns]
    missing_cols   = [c for c in expected_cols if c not in df_atm.columns]
    if missing_cols:
        print(f'  ⚠️  Missing columns: {missing_cols}')
    print(df_atm[available_cols].to_string(index=False))

# Thunderstorm cells (WMO code)
WMO_THUNDER = {95:'light thunderstorm',96:'hail',99:'heavy thunderstorm'}
thunder_points = df_atm[df_atm['thunderstorm'] == True]
if not thunder_points.empty:
    print(f'\nActive thunderstorm cells: {len(thunder_points)}')
    for _, row in thunder_points.iterrows():
        wc = int(row['wcode']) if pd.notna(row['wcode']) else 0
        print(f'  {row["point"]}: WMO {wc} – {WMO_THUNDER.get(wc, "Thunderstorm")}')
else:
    print('\nNo active thunderstorm cells at measurement points')

# --- Active storm events (NASA EONET) ---
storm_events = []
if raw['storms']:
    for ev in raw['storms'].get('events', []):
        geom = ev.get('geometry', [])
        last = geom[-1] if geom else {}
        coords = last.get('coordinates', [None, None])
        storm_events.append({
            'title':   ev.get('title', ''),
            'date':    last.get('date', ''),
            'lon':     coords[0] if len(coords) > 1 else None,
            'lat':     coords[1] if len(coords) > 1 else None,
        })
    print(f'\nNASA EONET storms: {len(storm_events)} active events')
    for s in storm_events[:5]:
        print(f'  {s["date"][:10]}  {s["title"]}')

# --- Kp current ---
kp_now = None
if raw['kp']:
    df_kp = pd.DataFrame(raw['kp'])
    tc = next((c for c in df_kp.columns if 'time' in c.lower()), df_kp.columns[0])
    kc = 'kp' if 'kp' in df_kp.columns else next(
        (c for c in df_kp.columns if 'kp' in c.lower() and c != tc), df_kp.columns[1])
    df_kp[kc] = df_kp[kc].astype(str).str.extract(r'([0-9]+(?:\.[0-9]*)?)')[0]
    df_kp[kc] = pd.to_numeric(df_kp[kc], errors='coerce')
    df_kp = df_kp[df_kp[kc] >= 0].dropna()
    if not df_kp.empty:
        kp_now = float(df_kp[kc].iloc[-1])
        print(f'Kp current: {kp_now:.1f}')

print('\nData processing complete')



Atmospheric parameters:
          point  pressure  temp_2m   cape   cin   li  cloud  thunderstorm
 Central Europe     937.0      7.4   10.0   0.0 10.5    100         False
Tropics (Congo)     944.9     31.9 1610.0   1.0 -2.3     25         False
         Arctic     944.2     -4.7    0.0   0.0 14.1     95         False
 Pacific (ITCZ)    1006.5     28.3 2230.0   2.0 -4.0     47         False
       Amazonia    1011.6     23.2  210.0   0.0 -1.7    100         False
          Sahel     971.2     41.3    0.0 152.0  8.6      0         False

No active thunderstorm cells at measurement points

NASA EONET storms: 1 active events
  2026-05-10  Tropical Storm Hagupit
Kp current: 0.0

Data processing complete


---
## 3. Visualizations

In [5]:
# ============================================================
# CAPE / CIN / LIFTED INDEX – Convection potential
# ============================================================

if not df_atm.empty:
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=['CAPE [J/kg]', 'CIN [J/kg]', 'Lifted Index [K]'],
        horizontal_spacing=0.1
    )
    pts = df_atm['point'].tolist()
    pt_colors = ['#E85D24','#185FA5','#888780','#378ADD','#639922','#F2A623']

    # CAPE – höher = mehr Gewitterpotential
    cape_vals = df_atm['cape'].fillna(0).tolist()
    cape_colors = ['#e74c3c' if v > 2000 else '#f39c12' if v > 500 else '#2ecc71'
                   for v in cape_vals]
    fig.add_trace(go.Bar(x=pts, y=cape_vals, marker_color=cape_colors,
                         opacity=0.85, showlegend=False), row=1, col=1)
    fig.add_hline(y=2000, line_dash='dot', line_color='#e74c3c',
                  annotation_text='High', row=1, col=1)
    fig.add_hline(y=500,  line_dash='dot', line_color='#f39c12',
                  annotation_text='Moderate', row=1, col=1)

    # CIN – negativer = stärker gedeckelt (weniger Auslöse)
    cin_vals = df_atm['cin'].fillna(0).tolist()
    cin_colors = ['#e74c3c' if v < -200 else '#f39c12' if v < -50 else '#2ecc71'
                  for v in cin_vals]
    fig.add_trace(go.Bar(x=pts, y=cin_vals, marker_color=cin_colors,
                         opacity=0.85, showlegend=False), row=1, col=2)

    # Lifted Index – negativ = instabil
    li_vals = df_atm['li'].fillna(0).tolist()
    li_colors = ['#e74c3c' if v < -4 else '#f39c12' if v < 0 else '#2ecc71'
                 for v in li_vals]
    fig.add_trace(go.Bar(x=pts, y=li_vals, marker_color=li_colors,
                         opacity=0.85, showlegend=False), row=1, col=3)
    fig.add_hline(y=0, line_color='gray', line_width=0.7, row=1, col=3)
    fig.add_hline(y=-4, line_dash='dot', line_color='#e74c3c',
                  annotation_text='Strongly Unstable', row=1, col=3)

    fig.update_layout(
        title=dict(text='Convection Parameters (Open-Meteo, 6 Reference Points)', font=dict(size=14)),
        height=380, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=50, r=30, t=60, b=80)
    )
    fig.update_xaxes(tickangle=30)
    fig.show()

In [6]:
# ============================================================
# LIGHTNING CHAIN: Convection → Thunderstorms → Schumann
# Shows the causal chain per measurement point as heatmap
# ============================================================

def cape_score(v):
    """CAPE → normalized score [0,1]"""
    if v is None or pd.isna(v): return 0.0
    return min(1.0, float(v) / 3000)

def li_score(v):
    """Lifted Index: negative = unstable → score [0,1]"""
    if v is None or pd.isna(v): return 0.3
    return min(1.0, max(0.0, (-float(v) + 6) / 12))

def cloud_score(v):
    if v is None or pd.isna(v): return 0.3
    return float(v) / 100

def thunder_score(row):
    """Kombinierter Gewitterwahrscheinlichkeits-Score
    CAPE + LI + Bedeckung + WMO-Gewitter + konvektiver Niederschlag (showers)
    Hinweis: X-Ray und Kp sind kein Bestandteil – nur geomagn./solarer Kontext.
    """
    cs = cape_score(row.get('cape'))
    ls = li_score(row.get('li'))
    cl = cloud_score(row.get('cloud'))
    t  = 1.0 if row.get('thunderstorm') else 0.0
    # Convective precipitation (showers) as additional indicator
    sh = min(1.0, (row.get('showers') or 0) / 5.0)
    return round(cs * 0.30 + ls * 0.25 + cl * 0.15 + t * 0.20 + sh * 0.10, 3)

def schumann_contribution(t_score, lat):
    """Schumann contribution: higher in tropics (±30°), scaled by thunderstorm score"""
    lat_weight = max(0.3, 1.0 - abs(lat) / 60)
    return round(t_score * lat_weight, 3)

df_atm['cape_score']   = df_atm.apply(lambda r: cape_score(r['cape']), axis=1)
df_atm['li_score']     = df_atm.apply(lambda r: li_score(r['li']),    axis=1)
df_atm['thunder_score']= df_atm.apply(lambda r: thunder_score(r.to_dict()), axis=1)
df_atm['schumann_contribution'] = df_atm.apply(
    lambda r: schumann_contribution(r['thunder_score'], r['lat']), axis=1)

metrics = ['cape_score', 'li_score', 'thunder_score', 'schumann_contribution']
labels  = ['CAPE-Score', 'LI-Score\n(Instability)', 'Thunder-\nScore', 'Schumann-\nContrib.']
z_data  = [df_atm[m].tolist() for m in metrics]

fig = go.Figure(go.Heatmap(
    z=z_data,
    x=df_atm['point'].tolist(),
    y=labels,
    colorscale='YlOrRd',
    zmin=0, zmax=1,
    text=[[f'{v:.2f}' for v in row] for row in z_data],
    texttemplate='%{text}', textfont=dict(size=12),
    colorbar=dict(title='Score [0-1]')
))
fig.update_layout(
    title=dict(text='Lightning Chain: Convection → Thunderstorms → Schumann Contribution', font=dict(size=14)),
    height=310,
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=130, r=80, t=55, b=80)
)
fig.show()


In [7]:
# ============================================================
# STORM EVENTS – World map (NASA EONET + WMO thunderstorm points)
# ============================================================

fig = go.Figure()

# NASA EONET storms
if storm_events:
    storm_lons = [s['lon'] for s in storm_events if s['lon'] is not None]
    storm_lats = [s['lat'] for s in storm_events if s['lat'] is not None]
    storm_texts = [f"{s['title']}<br>{s['date'][:10]}" for s in storm_events if s['lon'] is not None]
    fig.add_trace(go.Scattergeo(
        lon=storm_lons, lat=storm_lats, mode='markers',
        marker=dict(size=14, color='#E85D24', symbol='x', opacity=0.85,
                    line=dict(color='white', width=1)),
        text=storm_texts,
        hovertemplate='%{text}<extra>NASA EONET Storm</extra>',
        name='NASA EONET Storms'
    ))

# WMO thunderstorm points from Open-Meteo
thunder_df = df_atm[df_atm['thunderstorm'] == True]
if not thunder_df.empty:
    fig.add_trace(go.Scattergeo(
        lon=thunder_df['lon'], lat=thunder_df['lat'], mode='markers',
        marker=dict(size=16, color='#F2A623', symbol='star', opacity=0.9),
        text=thunder_df['point'],
        hovertemplate='%{text}<br>Thunderstorm (WMO)<extra></extra>',
        name='WMO Thunderstorm Points'
    ))

# All measurement points (convection strength as size)
fig.add_trace(go.Scattergeo(
    lon=df_atm['lon'], lat=df_atm['lat'], mode='markers+text',
    marker=dict(
        size=[max(8, v * 30) for v in df_atm['thunder_score']],
        color=df_atm['thunder_score'].tolist(),
        colorscale='YlOrRd', cmin=0, cmax=1, opacity=0.7,
        colorbar=dict(title='Thunder Score', x=1.0)
    ),
    text=df_atm['point'],
    textposition='top center',
    textfont=dict(size=9, color='white'),
    hovertemplate='%{text}<br>Score: %{marker.color:.2f}<extra></extra>',
    name='Measurement Points'
))

fig.update_geos(
    projection_type='natural earth',
    showland=True, landcolor='#2C2C2A',
    showocean=True, oceancolor='#0C2040',
    showcoastlines=True, coastlinecolor='#444441',
    showframe=False
)
fig.update_layout(
    title=dict(text='Global Thunderstorm Activity – Layer 3', font=dict(size=14)),
    height=430, margin=dict(l=0, r=0, t=50, b=0),
    paper_bgcolor='rgba(0,0,0,0)',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(30,30,30,0.7)',
                font=dict(color='white', size=10))
)
fig.show()

In [8]:
# ============================================================
# X-RAY FLUX (GOES) – Solar ionization as atmospheric modulator
# ============================================================

if raw['xray']:
    df_x = pd.DataFrame(raw['xray'])
    tc   = next((c for c in df_x.columns if 'time' in c.lower()), df_x.columns[0])
    fc   = next((c for c in df_x.columns
                 if any(k in c.lower() for k in ['flux','long','energy']) and c != tc),
                df_x.columns[1])
    df_x['time'] = pd.to_datetime(df_x[tc])
    df_x['flux'] = pd.to_numeric(df_x[fc], errors='coerce')
    df_x = df_x.dropna(subset=['flux']).sort_values('time')

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_x['time'], y=df_x['flux'],
        line=dict(color='#E85D24', width=1.5), name='X-Ray Flux'
    ))
    for lvl, lbl, col in [
        (1e-4,'X','#c0392b'), (1e-5,'M','#e67e22'),
        (1e-6,'C','#f1c40f'), (1e-7,'B','#2ecc71')
    ]:
        fig.add_hline(y=lvl, line_dash='dot', line_color=col,
                      annotation_text=lbl)
    fig.update_layout(
        title=dict(text='GOES X-Ray Flux – Solar Ionization (Atmospheric Modulator)', font=dict(size=13)),
        yaxis=dict(type='log', title='Flux [W/m²]'),
        height=300, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=70, r=30, t=50, b=40), showlegend=False
    )
    fig.show()

---
## 4. State Assessment & Handoff to Layer 4/5

In [9]:
# ============================================================
# LAYER-3-SCORE
# Five dynamic components
# ============================================================

def norm(v, lo, hi):
    if v is None or (isinstance(v, float) and math.isnan(v)): return None
    return round(max(0.0, min(1.0, (v - lo) / (hi - lo))), 4)

# --- 1) Convection intensity (global CAPE mean) ---
cape_mean = df_atm['cape'].dropna().mean() if not df_atm.empty else None
conv_score = norm(cape_mean, 0, 3000)
conv_src   = 'primary' if cape_mean is not None else 'missing'

# --- 2) Atmospheric instability (LI mean) ---
li_mean   = df_atm['li'].dropna().mean() if not df_atm.empty else None
# Negative LI = unstable → higher score
instab_score = norm(-li_mean if li_mean is not None else None, -6, 6)
instab_src   = 'primary' if li_mean is not None else 'missing'

# --- 3) Thunderstorm activity (mean thunder score + WMO count) ---
thunder_mean = df_atm['thunder_score'].mean() if not df_atm.empty else None
n_thunder    = int(df_atm['thunderstorm'].sum()) if not df_atm.empty else 0
thunder_score_val = norm(
    (thunder_mean or 0) * 0.7 + n_thunder * 0.1, 0, 1.0
) if thunder_mean is not None else None
thunder_src = 'primary'

# --- 4) Schumann excitation potential (weighted Schumann contribution) ---
schumann_pot = df_atm['schumann_contribution'].mean() if not df_atm.empty else None
# Boosted by Layer-2 thunderstorm trigger
if schumann_pot is not None and l2_thunder:
    schumann_pot = min(1.0, schumann_pot * 1.2)
schumann_score = norm(schumann_pot, 0, 1.0) if schumann_pot is not None else None
schumann_src   = 'derived'

# --- 5) Storm activity (NASA EONET) ---
# Distinction: None = data not loaded, 0.0 = loaded but no events
storm_score = None
storm_src   = 'missing'
if raw.get('storms') is not None:          # API reached successfully
    storm_score = norm(len(storm_events), 0, 30) if storm_events else 0.0
    storm_src   = 'primary'

# X-Ray and Kp: context only, not in dynamic score
# (X-Ray = Layer-0/4 driver; Kp = geomagnetic index, not a lightning proxy)
xray_context = None
if raw.get('xray'):
    df_xc = pd.DataFrame(raw['xray'])
    fc = next((c for c in df_xc.columns
               if any(k in c.lower() for k in ['flux','long']) and 'time' not in c.lower()),
              df_xc.columns[1])
    df_xc[fc] = pd.to_numeric(df_xc[fc], errors='coerce')
    xray_last = df_xc[fc].dropna().iloc[-1] if not df_xc[fc].dropna().empty else None
    if xray_last:
        xray_class = ('X' if xray_last>=1e-4 else 'M' if xray_last>=1e-5
                      else 'C' if xray_last>=1e-6 else 'B')
        xray_context = {'flux': float(xray_last), 'class': xray_class}
        print(f'X-Ray context: {xray_class}-class ({xray_last:.2e} W/m²)')

# --- Aggregate ---
COMPONENTS = {
    'Convection Intensity (CAPE)':  {'score': conv_score,       'source': conv_src,    'dynamic': True},
    'Atm. Instability (LI)':         {'score': instab_score,     'source': instab_src,  'dynamic': True},
    'Thunderstorm Activity':              {'score': thunder_score_val, 'source': thunder_src, 'dynamic': True},
    'Schumann Excitation Potential':   {'score': schumann_score,   'source': schumann_src,'dynamic': True},
    'Active Storm Events (EONET)': {'score': storm_score,      'source': storm_src,   'dynamic': True},
}

available   = {k: v['score'] for k, v in COMPONENTS.items() if v['score'] is not None}
unavailable = [k for k, v in COMPONENTS.items() if v['score'] is None]
layer3_score = round(sum(available.values()) / len(available), 4) if available else None
confidence   = round(len(available) / len(COMPONENTS), 2)
level = ('unknown' if layer3_score is None
         else 'quiet'   if layer3_score < 0.3
         else 'moderate' if layer3_score < 0.6
         else 'active')
dominant_l3 = max(available, key=available.get) if available else 'none'

# --- Ausgabe ---
W = 68
print('=' * W)
print('LAYER 3 – ATMOSPHERE / WEATHER / THUNDERSTORMS – STATE ASSESSMENT')
print('=' * W)
for name, comp in COMPONENTS.items():
    s = comp['score']
    if s is not None:
        bar = '█' * int(s * 20) + '░' * (20 - int(s * 20))
        print(f'  {name:<36} {bar}  {s:.3f}  [{comp["source"]}]')
    else:
        print(f'  {name:<36} {"─" * 20}  n/a   [missing]')
print('-' * W)
print(f'  Score:        {layer3_score:.3f}  ({len(available)}/{len(COMPONENTS)} components)')
print(f'  Confidence:   {confidence:.0%}')
print(f'  Level:        {level.upper()}')
print(f'  Dominant:     {dominant_l3}')
print(f'  CAPE mean:    {cape_mean:.0f} J/kg' if cape_mean else '  CAPE mean:    n/a')
print(f'  LI mean:      {li_mean:.1f} K' if li_mean is not None else '  LI mean:      n/a')
print(f'  Thunderstorms:{n_thunder} points (WMO)  +  {len(storm_events)} NASA EONET')
print('=' * W)

# Radar
cats   = list(available.keys())
vals_r = list(available.values())
if len(cats) >= 3:
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=vals_r + [vals_r[0]], theta=cats + [cats[0]],
        fill='toself', fillcolor='rgba(127,119,221,0.22)',
        line=dict(color='#7F77DD', width=2.5), name='Layer 3'
    ))
    fig.add_trace(go.Scatterpolar(
        r=[0.5] * (len(cats)+1), theta=cats + [cats[0]],
        line=dict(color='#E24B4A', dash='dot', width=1),
        mode='lines', name='Activity Threshold'
    ))
    fig.update_layout(
        title=dict(
            text=f'Layer 3 – Activity Profile | Score: {layer3_score:.3f} | {level.upper()} | Confidence: {confidence:.0%}',
            font=dict(size=13)
        ),
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        height=450, showlegend=True,
        margin=dict(l=60, r=60, t=65, b=40)
    )
    fig.show()



X-Ray context: B-class (6.03e-07 W/m²)
LAYER 3 – ATMOSPHERE / WEATHER / THUNDERSTORMS – STATE ASSESSMENT
  Convection Intensity (CAPE)          ████░░░░░░░░░░░░░░░░  0.226  [primary]
  Atm. Instability (LI)                ███░░░░░░░░░░░░░░░░░  0.150  [primary]
  Thunderstorm Activity                ███░░░░░░░░░░░░░░░░░  0.176  [primary]
  Schumann Excitation Potential        ████░░░░░░░░░░░░░░░░  0.201  [derived]
  Active Storm Events (EONET)          ░░░░░░░░░░░░░░░░░░░░  0.033  [primary]
--------------------------------------------------------------------
  Score:        0.157  (5/5 components)
  Confidence:   100%
  Level:        QUIET
  Dominant:     Convection Intensity (CAPE)
  CAPE mean:    677 J/kg
  LI mean:      4.2 K
  Thunderstorms:0 points (WMO)  +  1 NASA EONET


In [10]:
# ============================================================
# EXPORT – layer3_test_state.json
# ============================================================

# Schumann downstream assessment
_schumann = (
    'strong Schumann excitation expected – high thunderstorm and CAPE level'
    if layer3_score is not None and layer3_score > 0.65
    else 'moderate Schumann excitation possible – convection active'
    if layer3_score is not None and layer3_score > 0.35
    else 'low Schumann excitation – quiet convection'
)

# GEC input: lightning flashes per minute worldwide (climatology reference: ~45/s)
# Approximation: thunderstorm score → relative GEC input
_gec_input = (
    'elevated lightning current input into GEC'
    if thunder_score_val is not None and thunder_score_val > 0.5
    else 'normal GEC baseline state'
)

# state_summary
_cape_str  = f'CAPE-Mittel {cape_mean:.0f} J/kg.' if cape_mean is not None else 'CAPE not available.'
_li_str    = f'LI-Mittel {li_mean:.1f} K.' if li_mean is not None else 'LI not available.'
_thun_str  = f'{n_thunder} WMO thunderstorm points, {len(storm_events)} NASA storm events.'
_l2_str    = (f'L2 input: {layer2["level"]} (Conv={l2_conv:.2f}, Thunder-Trigger={l2_thunder}).' if layer2
              else 'L2 context missing.')

state_summary = ' '.join([
    f'Layer-3-State: {level}.', _cape_str, _li_str, _thun_str,
    f'Schumann potential: {schumann_score:.3f}.' if schumann_score is not None else '',
    f'Data completeness: {confidence:.0%}.', _l2_str
]).strip()

layer3_state = {
    'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
    'layer': 3,
    'name':  'Atmosphere / Weather / Thunderstorms',

    'score':      layer3_score,
    'level':      level,
    'confidence': confidence,
    'score_basis': f'{len(available)}/{len(COMPONENTS)} dynamic components',
    'dominant_component': dominant_l3,
    'missing_components': unavailable,

    'components': {
        k: {
            'score':   round(v['score'], 4) if v['score'] is not None else None,
            'source':  v['source'],
            'dynamic': v['dynamic'],
        }
        for k, v in COMPONENTS.items()
    },

    'raw_values': {
        'CAPE_mean_Jkg':      round(cape_mean, 1) if cape_mean is not None else None,
        'LI_mean_K':          round(li_mean, 2)   if li_mean   is not None else None,
        'thunder_points_WMO': n_thunder,
        'storm_events_EONET': len(storm_events),
        'Kp_current':         round(kp_now, 1)    if kp_now    is not None else None,
        'xray_context':       xray_context,  # solarer Kontext (Layer-0/4-Treiber)
        'kp_note':            'geomagnetic context, not a lightning proxy',
        'schumann_potential': round(schumann_score, 4) if schumann_score is not None else None,
        'messpunkte': [
            {
                'point':       r['point'],
                'cape':   (float(r['cape']) if r.get('cape') == r.get('cape') and r.get('cape') is not None else None),
                'li':    (float(r['li']) if r.get('li') == r.get('li') and r.get('li') is not None else None),
                'thunderstorm': bool(r.get('thunderstorm', False)),
                'thunder_score': (float(r['thunder_score']) if r.get('thunder_score') is not None else None),
                'schumann_contribution': (float(r['schumann_contribution']) if r.get('schumann_contribution') is not None else None),
            }
            for r in df_atm.to_dict('records')
        ]
    },

    'flags': {
        'active_thunderstorms':    bool(n_thunder > 0),
        'high_cape':               bool(cape_mean > 2000)       if cape_mean is not None else None,
        'atmospheric_instability': bool(instab_score > 0.6)     if instab_score is not None else None,
        'elevated_storm_activity': bool(len(storm_events) > 10),
        'schumann_trigger':        bool(schumann_score > 0.5)   if schumann_score is not None else None,
        'l2_thunder_confirmed':    bool(l2_thunder),
    },

    'downstream_expectation': {
        'layer4_ionosphere': (
            'Thunderstorm columns reach tropopause – TIE effects and sprites possible'
            if n_thunder > 0
            else 'normal tropospheric influence on ionosphere'
        ),
        'layer5_gec': _gec_input,
        'schumann_resonance': _schumann,
        'layer6_pattern': (
            'Convection patterns active – atmospheric resonance structure modulated'
            if layer3_score is not None and layer3_score > 0.4
            else 'quiet atmosphere – no strong pattern modulation'
        ),
    },

    'layer0_context': {
        'score':           layer0['score']           if layer0 else None,
        'level':           layer0['level']           if layer0 else None,
        'dominant_driver': layer0.get('dominant_driver') if layer0 else None,
    },
    'layer1_context': {
        'score': layer1['score'] if layer1 else None,
        'level': layer1['level'] if layer1 else None,
    },
    'layer2_context': {
        'score':             layer2['score']                                      if layer2 else None,
        'level':             layer2['level']                                      if layer2 else None,
        'thunderstorm_trigger': l2_thunder,
        'conv_potential':    l2_conv,
        'enso_phase':        layer2.get('raw_values',{}).get('ENSO',{}).get('phase_observed') if layer2 else None,
    },

    'state_summary': state_summary,
}

# Sicherstellen dass alle Werte JSON-serialisierbar sind
def _to_python(obj):
    import numpy as np
    if isinstance(obj, dict):  return {k: _to_python(v) for k,v in obj.items()}
    if isinstance(obj, list):  return [_to_python(v) for v in obj]
    if isinstance(obj, (np.bool_,)):   return bool(obj)
    if isinstance(obj, (np.integer,)):  return int(obj)
    if isinstance(obj, (np.floating,)): return None if np.isnan(obj) else float(obj)
    return obj
layer3_state = _to_python(layer3_state)

with open('layer3_test_state.json', 'w', encoding='utf-8') as f:
    json.dump(layer3_state, f, indent=2, ensure_ascii=False)

print('layer3_test_state.json saved')
print(json.dumps(layer3_state, indent=2, ensure_ascii=False))




layer3_test_state.json saved
{
  "timestamp": "2026-05-12T12:28:37.156747Z",
  "layer": 3,
  "name": "Atmosphere / Weather / Thunderstorms",
  "score": 0.1571,
  "level": "quiet",
  "confidence": 1.0,
  "score_basis": "5/5 dynamic components",
  "dominant_component": "Convection Intensity (CAPE)",
  "missing_components": [],
  "components": {
    "Convection Intensity (CAPE)": {
      "score": 0.2256,
      "source": "primary",
      "dynamic": true
    },
    "Atm. Instability (LI)": {
      "score": 0.15,
      "source": "primary",
      "dynamic": true
    },
    "Thunderstorm Activity": {
      "score": 0.1756,
      "source": "primary",
      "dynamic": true
    },
    "Schumann Excitation Potential": {
      "score": 0.2008,
      "source": "derived",
      "dynamic": true
    },
    "Active Storm Events (EONET)": {
      "score": 0.0333,
      "source": "primary",
      "dynamic": true
    }
  },
  "raw_values": {
    "CAPE_mean_Jkg": 676.7,
    "LI_mean_K": 4.2,
    "thunder_po

---
## Summary Layer 3

| Aspekt | Inhalt |
|--------|--------|
| **Role** | Primary Schumann exciter – convection, thunderstorms, lightning activity |
| **Data sources** | Open-Meteo (CAPE/CIN/LI), NASA EONET, NOAA Kp, GOES X-Ray |
| **Key parameters** | CAPE, Lifted Index, WMO weather code, Schumann contribution |
| **Lightning chain** | CAPE → LI instability → thunderstorm score → Schumann contribution |
| **→ Layer 4** | Gewittersäulen → TIE-Effekte, Sprites, Ionospheren-Modifikation |
| **→ Layer 5** | Lightning current input into Global Electric Circuit |
| **→ Layer 6** | Convection patterns → Schumann resonance excitation |
| **Output** | `layer3_atmosphere_zone.json` with full L0–L2 context |

> **Nächster Schritt:** `layer4_Ionosphere.ipynb` – Ionosphere